# 02 · Forecasting Battle

**Forecast vs Instinct** - real data from **Corporación Favorita** (Ecuador, Kaggle *Store Sales - Time Series
Forecasting*), **12 stores × 8 product families = 96 daily series**, forecast **14 days ahead** from weekly origins.

This notebook tells the modelling story: how the data was split so the final test is honest, how look-ahead was made
impossible, which five models competed, how each was tuned, how a champion was picked per family, and how everything
did on the **holdout, which was evaluated exactly once** after all choices were frozen.

Ground rules (same as notebook 01):

1. **Sales are observed sales, not unconstrained demand.** A day with empty shelves looks like low demand.
2. **`onpromotion` is a count of promoted items**, with no price or discount depth. Promo comparisons are associations.
3. **Nothing is re-trained or re-run here.** Every number is computed in a cell from the saved backtests in
   `outputs/backtests/` and the frozen JSON files in `outputs/`. The notebook never writes to `outputs/`.

**Metrics** (`src/metrics.py`, A = actual, F = forecast, sums over the rows being scored):

| Metric | Formula | Read it as |
|---|---|---|
| WAPE | Σ\|A − F\| / ΣA × 100 | % of volume mis-forecast; **lower is better**; the headline metric |
| Bias | Σ(F − A) / ΣA × 100 | **+ = over-forecast**, − = under-forecast |
| MAE | mean \|A − F\| | average miss per series-day, in units |
| RMSE | √mean (A − F)² | like MAE but punishes big misses more |

WAPE and bias are always **ratio of sums** over the group (never an average of per-row percentages), and every model
is scored on the identical set of rows: open days with an observed actual (`src.backtest.evaluable`).

In [1]:
import sys, json, hashlib, warnings
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from src import config as C, metrics as M
from src.backtest import evaluable, masked_view
from src.runner import load_backtests
from src.selection import selection_table, pick_champions, champion_rows
from src import indexing as IX  # per-series values are shown as an index (Kaggle rule 7B: no raw data)

pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.precision", 3)
pd.set_option("display.width", 160)

OUT = C.OUTPUTS
J = lambda name: json.loads((OUT / name).read_text())

# Same family colours as notebook 01; one fixed colour per model (never re-assigned)
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
FAM_COLOR = dict(zip(C.FAMILIES, PALETTE))
MODEL_ORDER = ["naive", "seasonal_naive", "moving_average", "holt_winters", "lightgbm A", "lightgbm B"]
MODEL_COLOR = {"naive": "#8a8984", "seasonal_naive": "#eda100", "moving_average": "#e87ba4",
               "holt_winters": "#2a78d6", "lightgbm A": "#eb6834", "lightgbm B": "#6250d6"}
BAND_COLOR = {"Accept": "#1baf7a", "Monitor": "#eda100", "Review": "#e34948"}
LAYOUT = dict(template="simple_white", height=380, margin=dict(l=50, r=20, t=50, b=40),
              font=dict(size=12), legend=dict(title=None))

bt = load_backtests()                                   # every model, every stage, read-only
bt["mv"] = np.where(bt["model"] == "lightgbm", "lightgbm " + bt["variant"], bt["model"])
panel = pd.read_parquet(C.PROCESSED / "panel.parquet")
cal = json.loads((C.CONFIG / "split_calendar.json").read_text())

def table(df, by, models=MODEL_ORDER, value="wape"):
    """Ratio-of-sums metric per group x model (src.metrics.grouped_metrics), models as columns."""
    g = M.grouped_metrics(df, list(by) + ["mv"])
    return g.pivot_table(index=list(by), columns="mv", values=value)[[m for m in models if m in set(g.mv)]]

print(f"backtest rows: {len(bt):,}  |  stages: {sorted(bt.stage.unique())}  |  models: {sorted(bt.mv.unique())}")

backtest rows: 201,600  |  stages: ['calibration', 'forward', 'holdout', 'selection']  |  models: ['holt_winters', 'lightgbm A', 'lightgbm B', 'moving_average', 'naive', 'seasonal_naive']


## 1. The split calendar: three stages, one honest test

A **forecast origin T** is the end of day T: everything up to and including T is known, and we forecast T+1 … T+14.
Origins are weekly (Tuesdays). They are grouped into three stages, each with one job:

| Stage | Job | What is decided there |
|---|---|---|
| **selection** | pick and tune | moving-average window, Holt-Winters trend, LightGBM settings, champion model per family |
| **calibration** | measure uncertainty | prediction-interval quantiles, Trust Index weights |
| **holdout** | final exam | nothing - it is scored once, after the choices above are frozen |

The **target windows** (the dates being forecast) of the three stages never overlap, so no date that is graded in
the holdout was ever used to make a choice.

In [2]:
rows = []
for stage, origins in cal["stages"].items():
    o = pd.to_datetime(origins)
    rows.append({"stage": stage, "n_origins": len(o), "first_origin": o.min().date(), "last_origin": o.max().date(),
                 "first_target": (o.min() + pd.Timedelta(days=1)).date(),
                 "last_target": (o.max() + pd.Timedelta(days=cal["horizon"])).date()})
split = pd.DataFrame(rows)
for a, b in zip(split.itertuples(), split.iloc[1:].itertuples()):
    gap = (pd.Timestamp(b.first_target) - pd.Timestamp(a.last_target)).days
    assert gap >= 1, "target windows overlap"
    print(f"{a.stage:>11} targets end {a.last_target}  ->  {b.stage} targets start {b.first_target}  (disjoint)")
# the saved backtests use exactly these origins
for stage, origins in cal["stages"].items():
    assert sorted(bt.loc[bt.stage == stage, "origin"].dt.strftime("%Y-%m-%d").unique()) == origins
print("Saved backtests use exactly the calendar origins for every stage.")
split

  selection targets end 2017-04-25  ->  calibration targets start 2017-04-26  (disjoint)
calibration targets end 2017-06-27  ->  holdout targets start 2017-06-28  (disjoint)
Saved backtests use exactly the calendar origins for every stage.


,stage,n_origins,first_origin,last_origin,first_target,last_target
0,selection,10,2017-02-07,2017-04-11,2017-02-08,2017-04-25
1,calibration,8,2017-04-25,2017-06-13,2017-04-26,2017-06-27
2,holdout,6,2017-06-27,2017-08-01,2017-06-28,2017-08-15


In [3]:
win = pd.DataFrame([{"stage": s, "origin": o, "start": pd.Timestamp(o) + pd.Timedelta(days=1),
                     "end": pd.Timestamp(o) + pd.Timedelta(days=cal["horizon"] + 1)}
                    for s, os_ in cal["stages"].items() for o in os_])
fig = px.timeline(win, x_start="start", x_end="end", y="origin", color="stage",
                  color_discrete_map={"selection": "#2a78d6", "calibration": "#eda100", "holdout": "#e34948"},
                  title="Each bar = the 14 target days of one origin")
fig.update_yaxes(autorange="reversed", title=None, tickfont=dict(size=9))
fig.update_layout(**{**LAYOUT, "height": 480}, xaxis_title=None)
fig.show()

**Reading it:** within a stage the windows overlap (weekly origins, 14-day horizon), but there is a gap between
stages so the last selection target comes before the first calibration target, and the last calibration target
(27 June 2017) comes before the first holdout target (28 June).

### The holdout was touched once

Before the holdout ran, the three decision files (`selection.json`, `intervals.json`, `trust_weights.json`) were
**frozen with a SHA-256 hash**. The holdout script refuses to run if any file changed since, and it logs the hashes and
a digest of its results the first (and only) time it runs (`outputs/holdout_run.json`). The cell below recomputes
the hashes now and re-derives the results digest from `holdout_results.json`.

In [4]:
run = J("holdout_run.json")
hrows = []
for n in ["selection", "intervals", "trust_weights"]:
    now = hashlib.sha256((OUT / f"{n}.json").read_bytes()).hexdigest()
    frozen = (OUT / f"{n}.sha256").read_text().strip()
    hrows.append({"file": f"{n}.json", "frozen hash": frozen[:16] + "…", "hash logged by holdout run": run["frozen_hashes"][n][:16] + "…",
                  "recomputed now": now[:16] + "…", "all three match": now == frozen == run["frozen_hashes"][n]})
display(pd.DataFrame(hrows))
res = J("holdout_results.json")
digest = hashlib.sha256(json.dumps({k: res[k] for k in ("leaderboard", "champion_system", "trust_validation")},
                                   sort_keys=True, default=float).encode()).hexdigest()
print("Holdout first (and only) run at:", run["first_run_at"])
print("Results digest logged at that run matches holdout_results.json:", digest == run["results_digest"])

,file,frozen hash,hash logged by holdout run,recomputed now,all three match
0,selection.json,c1ae56f4b398bba9…,c1ae56f4b398bba9…,c1ae56f4b398bba9…,True
1,intervals.json,bfc2c7459b8a0591…,bfc2c7459b8a0591…,bfc2c7459b8a0591…,True
2,trust_weights.json,dc75cfb2f8a7c09e…,dc75cfb2f8a7c09e…,dc75cfb2f8a7c09e…,True


Holdout first (and only) run at: 2026-11-25T21:04:43+00:00
Results digest logged at that run matches holdout_results.json: True


## 2. Leakage controls: making look-ahead impossible

"Leakage" means a model quietly uses information it would not have had on the day the forecast was made. Three
controls stop it:

**1. Masked view.** A model never receives the raw panel. `src.backtest.masked_view(panel, T, variant)` hands it a
copy where every **sale after T is blanked (NaN)**. In **promo variant B** the promoted-item count after T is blanked
too; in **variant A** it is kept, as a planner with an agreed promo calendar would know it. The demo below uses the
first selection origin.

Raw competition data is not redistributed, so the demo shows **indexed** values: sales as an index (÷ the series'
mean daily sales on open days 2015-06-01..2017-02-06 × 100, so 100 = an average training-period day) and promotions
as a promo index (÷ the series' mean promoted-item count from 2015-06-01 up to T × 100). Masking is applied to the
raw panel first and the result is indexed afterwards, so a blanked value stays NaN.

In [5]:
T0 = pd.Timestamp(cal["stages"]["selection"][0])
sid = "12|BEVERAGES"
one = panel[panel.series_id == sid]
sb, pb = IX.series_base(panel)[sid], IX.promo_base(panel, T0)[sid]   # private bases: used, never printed
around = lambda d: d[(d.date > T0 - pd.Timedelta(days=3)) & (d.date <= T0 + pd.Timedelta(days=3))].set_index("date")
si = lambda d: IX.to_index(around(d)["sales"], sb).round(1)
pi = lambda d: IX.to_index(around(d)["onpromotion"], pb).round(1)
demo = pd.DataFrame({
    "sales index (panel)": si(one), "promo index (panel)": pi(one),
    "view A sales": si(masked_view(one, T0, "A")), "view A promo": pi(masked_view(one, T0, "A")),
    "view B sales": si(masked_view(one, T0, "B")), "view B promo": pi(masked_view(one, T0, "B"))})
demo.index = [f"{d:%Y-%m-%d}" + ("  <- origin T" if d == T0 else "") for d in demo.index]
print(f"Series {sid}, origin T = {T0:%Y-%m-%d}; index values (100 = the series' average training-period day)")
demo

Series 12|BEVERAGES, origin T = 2017-02-07; index values (100 = the series' average training-period day)


,sales index (panel),promo index (panel),view A sales,view A promo,view B sales,view B promo
2017-02-05,133.8,145.8,133.8,145.8,133.8,145.8
2017-02-06,84.7,167.6,84.7,167.6,84.7,167.6
2017-02-07 <- origin T,84.0,0.0,84.0,0.0,84.0,0.0
2017-02-08,78.3,167.6,NaN,167.6,NaN,NaN
2017-02-09,75.1,80.2,NaN,80.2,NaN,NaN
2017-02-10,79.0,58.3,NaN,58.3,NaN,NaN


**2. Direct multi-horizon forecasting.** LightGBM does not forecast day 1 and then feed that guess back in to forecast
day 2 (a *recursive* model, where errors compound). It is one global model with **`horizon` (1-14) as a feature**,
trained on real pairs "history at origin t' → actual sale at t' + h". Every horizon is predicted from the same
observed history at T.

**3. Origin-relative features.** Every history feature is measured **backwards from the origin**, never from the target
date: `lag_1` = sales on T itself, `roll_mean_28` = mean of the 28 days up to T, `sw_last` = the most recent day ≤ T on
the target's weekday. Target-date features are only things known in advance: calendar (weekday, day of month, month,
holiday / event / payday flags) and, in variant A only, the planned promo count. Transactions and oil are not used
(future transactions are unknown at T; oil was tested and dropped, section 5).

The project's tests (`tests/test_features_leakage.py`) perturb every value after T and check that the forecasts do not
change; a deliberately injected one-day look-ahead made 5 of them fail. The cell below confirms from the saved feature
lists that variant B has no target-date promo feature.

In [6]:
fi = J("lgbm_feature_importance.json")
featA = [r["feature"] for r in fi["variants"]["A"]]
featB = [r["feature"] for r in fi["variants"]["B"]]
print(f"variant A: {len(featA)} features | variant B: {len(featB)} features")
print("only in A (target-date promo plan):", sorted(set(featA) - set(featB)))
print("only in B:", sorted(set(featB) - set(featA)) or "none")

variant A: 35 features | variant B: 33 features
only in A (target-date promo plan): ['promo_t', 'promo_t_rel28']
only in B: none


## 3. The five contenders, explained simply

| Model | One-line idea | Strength | Weakness |
|---|---|---|---|
| **Naive** | tomorrow (and every day of the next 14) = today | zero effort, a floor | ignores the weekly rhythm completely |
| **Seasonal naive** | each day = the same weekday last week | captures the Sunday peak / Thursday trough | copies last week's noise and one-offs |
| **Moving average** | each day = the mean of the last *w* days | smooth, robust | flat line: no weekday shape |
| **Holt-Winters** | exponential smoothing of a level + a weekly pattern, fitted per series on the last 364 days | the classic planner's statistical model | one series at a time; knows nothing about promos or holidays |
| **LightGBM** | one gradient-boosted tree model across all 96 series, learning how sales relate to history, calendar and promo plan | shares patterns across series; uses known drivers | needs careful leakage control and tuning |

LightGBM predicts a **scale-free target**, sales ÷ (28-day mean at the origin + 1), and multiplies back afterwards, so
the trees learn *shape* (weekday, promo, holiday) while the level comes from recent history. It runs in two promo
variants: **A = known promo plan**, **B = no future promo information**.

## 4. Tuning the simple models (selection stage only)

Two baseline settings were chosen on the 10 selection origins by **lowest overall selection WAPE**: the moving-average
window and whether Holt-Winters should include a (damped) trend.

In [7]:
bs = J("baselines_summary.json")
ma = pd.DataFrame(bs["moving_average"]["candidates"]).T.rename_axis("window (days)")
ma["chosen"] = ma.index.astype(int) == bs["moving_average"]["chosen_window"]
hw = pd.DataFrame(bs["holt_winters"]["trend_candidates"]).T.rename_axis("Holt-Winters trend")
hw["chosen"] = hw.index == bs["holt_winters"]["chosen_trend"]
display(ma[["wape", "bias", "n", "chosen"]])
display(hw[["wape", "bias", "n", "fallbacks", "chosen"]])
print(f"Holt-Winters fits: {bs['holt_winters']['fits_total']:,}, fallbacks to seasonal naive: {bs['holt_winters']['fallbacks_total']}")

,wape,bias,n,chosen
window (days),,,,
7,20.778,0.879,13440.0,False
14,20.248,0.795,13440.0,False
28,19.683,0.377,13440.0,True
56,20.677,2.805,13440.0,False


,wape,bias,n,fallbacks,chosen
Holt-Winters trend,,,,,
none,14.726,-0.493,13440.0,0.0,True
add_damped,14.924,-0.387,13440.0,0.0,False


Holt-Winters fits: 1,728, fallbacks to seasonal naive: 0


**Reading it:** a 28-day window is the best of the four moving averages, and adding a damped trend made Holt-Winters
slightly *worse* on selection, so no trend was used. Every Holt-Winters fit converged (no fallbacks).

## 5. Tuning LightGBM (selection stage only)

LightGBM was tuned in stages on the 10 selection origins (variant A, training every 2nd day to save time), ranking
candidates by overall selection WAPE rounded to 2 dp (ties → smaller |bias|). Each stage kept the winner of the
previous one. Variant B reuses the chosen settings unchanged.

In [8]:
tun = J("lgbm_tuning.json")
cand = pd.DataFrame([{"stage": c["stage"], "candidate": c["name"], "selection WAPE": c["selection_wape"],
                      "selection bias": c["selection_bias"], "runtime (s)": c["runtime_s"]} for c in tun["candidates"]])
cand

,stage,candidate,selection WAPE,selection bias,runtime (s)
0,objective,tweedie_p1.2,12.299,-1.691,78.7
1,objective,tweedie_p1.5,12.302,-1.769,80.2
2,objective,l2_raw,12.474,-1.484,66.1
3,objective,l1_raw,12.283,-2.190,86.9
4,objective,l2_ratio28,12.039,-0.632,71.3
5,structure,l2_ratio28+leaves31,12.086,-0.707,58.0
6,structure,l2_ratio28+leaves127,11.914,-0.615,98.7
7,structure,l2_ratio28+min_leaf30,12.061,-0.629,67.2
8,structure,l2_ratio28+min_leaf300,12.046,-0.661,61.9
9,structure,l2_ratio28+lr0.03_n800,12.017,-0.712,120.2


In [9]:
d = tun["decisions"]
print(f"Objective stage winner : {d['objective_stage_winner']}  (L2 loss on sales / 28-day mean)")
print(f"Structure stage winner : {d['structure_stage_winner']}")
print(f"Earthquake month       : '{d['quake']['choice']}'  flag WAPE {d['quake']['flag_wape']:.2f} (bias {d['quake']['flag_bias']:+.2f}) "
      f"vs drop WAPE {d['quake']['drop_wape']:.2f} (bias {d['quake']['drop_bias']:+.2f})")
print(f"Oil price              : keep={d['oil']['keep']}  without {d['oil']['without_wape']:.2f} vs with {d['oil']['with_wape']:.2f}  [{d['oil']['rule']}]")
print(f"Training-origin stride : {d['origin_stride']['choice']}  stride 2 {d['origin_stride']['stride2_wape']:.2f} vs stride 1 {d['origin_stride']['stride1_wape']:.2f}")
pd.Series(tun["chosen_params"]).drop(["tweedie_variance_power", "num_threads"]).to_frame("chosen").T

Objective stage winner : l2_ratio28  (L2 loss on sales / 28-day mean)
Structure stage winner : l2_ratio28+leaves127
Earthquake month       : 'flag'  flag WAPE 11.91 (bias -0.62) vs drop WAPE 11.94 (bias -1.06)
Oil price              : keep=False  without 11.91 vs with 12.27  [keep only if selection WAPE (2 dp) improves]
Training-origin stride : 1  stride 2 11.91 vs stride 1 11.88


,objective,target,num_leaves,learning_rate,n_estimators,min_data_in_leaf,feature_fraction,bagging_fraction,bagging_freq,lambda_l2,quake,use_oil,origin_stride
chosen,regression,ratio,127,0.05,400,100,0.8,0.8,1,1.0,flag,False,1


**Reading it:**

- **The scale-free target was the biggest win.** Predicting sales relative to the recent 28-day level beat every
  raw-sales objective (Tweedie, L2, L1) and cut the under-forecast bias from about −1.5 to −2.2 % down to about −0.6 %.
- **Earthquake (16 Apr - 15 May 2016): flag it, don't delete it.** Keeping those training rows with a quake flag gave
  a slightly lower WAPE and a smaller bias than dropping them.
- **Oil: dropped.** Adding the lagged national oil price made selection WAPE clearly worse. One national price series
  carries no store-level information for a 14-day horizon, and trends invite spurious splits.
- Training on every day (stride 1) helped a little, so the final model uses it (twice the run time).

(`tweedie_variance_power` is still in the saved parameter set but has no effect with the L2 objective.)

## 6. Choosing a champion per family (selection stage only)

Rule (`src/selection.py`): for each family, the model with the **lowest selection WAPE** (ratio of sums over 10 origins
× 12 stores × 14 days = 1,680 rows) is champion; a tie within 0.05 pp goes to the smaller |bias|. LightGBM competes
with variant A. The cell recomputes the table from the saved backtests and checks it against the frozen
`selection.json`.

In [10]:
sel = J("selection.json")
st = selection_table(bt)                                   # variant A rows, selection stage
frozen_st = pd.DataFrame(sel["selection_table"])
chk = st.merge(frozen_st, on=["family", "model"], suffixes=("", "_frozen"))
print("max |recomputed - frozen| selection WAPE:", round(float((chk.wape - chk.wape_frozen).abs().max()), 4))
champs = sel["champions"]
assert {f: c["model"] for f, c in pick_champions(st).items()} == {f: c["model"] for f, c in champs.items()}
piv = st.pivot(index="family", columns="model", values="wape").loc[C.FAMILIES, C.MODELS]
piv["champion"] = [champs[f]["model"] for f in piv.index]
piv["runner-up"] = [piv.loc[f, C.MODELS].drop(champs[f]["model"]).idxmin() for f in piv.index]
piv["margin (pp)"] = [piv.loc[f, r] - piv.loc[f, champs[f]["model"]] for f, r in zip(piv.index, piv["runner-up"])]
piv.round(2)

max |recomputed - frozen| selection WAPE: 0.0


model,naive,seasonal_naive,moving_average,holt_winters,lightgbm,champion,runner-up,margin (pp)
family,,,,,,,,
BEVERAGES,26.11,18.68,20.81,16.04,13.81,lightgbm,holt_winters,2.24
DAIRY,21.00,14.73,16.95,12.38,10.63,lightgbm,holt_winters,1.74
BREAD/BAKERY,20.50,16.27,17.75,13.60,11.58,lightgbm,holt_winters,2.02
GROCERY I,22.01,17.65,17.44,15.08,11.38,lightgbm,holt_winters,3.70
FROZEN FOODS,34.31,24.94,29.46,31.81,20.50,lightgbm,seasonal_naive,4.44
CLEANING,24.43,20.35,20.23,17.33,12.77,lightgbm,holt_winters,4.56
PERSONAL CARE,32.75,26.61,25.71,23.90,14.86,lightgbm,holt_winters,9.03
PRODUCE,36.64,12.88,22.05,10.16,9.53,lightgbm,holt_winters,0.63


In [11]:
fig = px.imshow(piv[C.MODELS].round(1), text_auto=True, aspect="auto", color_continuous_scale="Blues",
                title="Selection WAPE % by family and model (lower = better)")
fig.update_layout(**{**LAYOUT, "height": 420}, coloraxis_colorbar=dict(title="WAPE"), xaxis_title=None, yaxis_title=None)
fig.show()

**Reading it:** LightGBM (variant A) wins selection in **all 8 families**, so the champion system is simply LightGBM A
everywhere. Holt-Winters is the runner-up in seven families; in FROZEN FOODS the runner-up is seasonal naive, because
Holt-Winters over-forecast that family badly on selection (see `baselines_summary.json`: bias about +9 %).
The margin column shows how much daylight there was: under 1 pp for PRODUCE, 4-5 pp for FROZEN FOODS and CLEANING,
and about 9 pp for PERSONAL CARE.

## 7. The holdout leaderboard (6 origins, run once)

All six model/variant combinations are scored on exactly the same 8,064 holdout rows (6 origins × 96 series × 14 days,
open days with an actual). The cell recomputes the leaderboard from the saved backtests and checks it against
`holdout_results.json`.

In [12]:
hold = evaluable(bt[bt.stage == "holdout"])
sizes = hold.groupby("mv").size()
assert sizes.nunique() == 1, "evaluation population differs between models"
lb = M.grouped_metrics(hold, "mv").set_index("mv").loc[MODEL_ORDER, ["wape", "bias", "mae", "rmse", "n"]].sort_values("wape")
saved = pd.DataFrame(res["leaderboard"]).assign(mv=lambda d: np.where(d.model == "lightgbm", "lightgbm " + d.variant, d.model)).set_index("mv")
print("max |recomputed - saved| WAPE:", float((lb.wape - saved.loc[lb.index, "wape"]).abs().max()))
best_base = lb.drop(["lightgbm A", "lightgbm B"]).wape.idxmin()
print(f"LightGBM A vs best simple model ({best_base}): {lb.loc[best_base,'wape'] - lb.loc['lightgbm A','wape']:.2f} pp lower WAPE, "
      f"{100 * (1 - lb.loc['lightgbm A','wape'] / lb.loc[best_base,'wape']):.0f}% less mis-forecast volume")
# equal-weighted over the 96 series (each series on its own index scale), so no series' volume weighs more
pre = panel[(panel.date <= cal["target_windows"]["calibration"][1]) & (panel.closed == 0) & panel.sales.notna()].copy()
pre["idx"] = IX.to_index(pre.sales, pre.series_id.map(IX.series_base(panel)))
sw = pre.groupby(["series_id", pre.date.dt.dayofweek == 1]).idx.mean().unstack()
tue = (sw[True] / pre.groupby("series_id").idx.mean()).mean()
print(f"Why naive under-forecasts: origins are Tuesdays, and a Tuesday sells {tue:.3f}x an average day (mean over the 96 "
      f"series, history up to the last calibration target), so repeating Tuesday for 14 days misses the weekend peak.")
lb.round(2)

max |recomputed - saved| WAPE: 0.0
LightGBM A vs best simple model (holt_winters): 4.66 pp lower WAPE, 27% less mis-forecast volume
Why naive under-forecasts: origins are Tuesdays, and a Tuesday sells 0.955x an average day (mean over the 96 series, history up to the last calibration target), so repeating Tuesday for 14 days misses the weekend peak.


,wape,bias,mae,rmse,n
mv,,,,,
lightgbm A,12.57,-0.46,208.52,512.45,8064
lightgbm B,12.75,-0.80,211.46,508.10,8064
holt_winters,17.23,1.66,285.80,606.67,8064
seasonal_naive,18.73,-0.03,310.64,701.53,8064
moving_average,22.03,0.25,365.46,694.67,8064
naive,30.18,-5.15,500.57,938.21,8064


In [13]:
fig = go.Figure(go.Bar(x=lb.wape.round(2), y=lb.index, orientation="h", marker_color=[MODEL_COLOR[m] for m in lb.index],
                       text=lb.wape.round(1), textposition="outside", hovertemplate="%{y}: WAPE %{x:.2f}%<extra></extra>"))
fig.update_layout(**{**LAYOUT, "height": 330}, title="Holdout WAPE % (6 origins × 96 series × 14 days)",
                  yaxis=dict(autorange="reversed"), xaxis_title="WAPE %", showlegend=False)
fig.show()

**Reading it:** LightGBM with the known promo plan is the clear winner on every metric except RMSE, where variant B is
marginally lower (RMSE is dominated by a few very large misses). The ranking of the simple models is the textbook one:
Holt-Winters > seasonal naive > moving average > naive. Naive is the only model with a sizeable bias (about −5 %):
it repeats the origin day, a Tuesday, which sells below an average day (printed above).

The champion system *is* LightGBM A in every family, so its holdout numbers equal the first row
(`holdout_results.json → champion_system` adds interval coverage, covered in notebook 03).

## 8. By family

In [14]:
fam = table(hold, ["family"]).loc[C.FAMILIES]
fam["best simple model"] = fam[C.MODELS[:-1]].idxmin(axis=1)
fam["gap to best simple (pp)"] = fam[C.MODELS[:-1]].min(axis=1) - fam["lightgbm A"]
fam["selection WAPE (A)"] = [champs[f]["selection_wape"] for f in fam.index]
fam.round(2)

mv,naive,seasonal_naive,moving_average,holt_winters,lightgbm A,lightgbm B,best simple model,gap to best simple (pp),selection WAPE (A)
family,,,,,,,,,
BEVERAGES,30.50,23.03,25.01,19.99,17.74,17.92,holt_winters,2.25,13.81
DAIRY,21.22,12.02,18.06,10.76,9.15,9.03,holt_winters,1.61,10.63
BREAD/BAKERY,21.75,14.22,18.09,11.98,10.92,10.69,holt_winters,1.07,11.58
GROCERY I,23.76,15.90,18.01,15.48,8.63,9.02,holt_winters,6.84,11.38
FROZEN FOODS,36.87,22.97,30.67,25.74,18.66,18.10,seasonal_naive,4.31,20.50
CLEANING,40.89,35.97,28.90,34.31,22.03,22.11,moving_average,6.87,12.77
PERSONAL CARE,36.97,25.46,27.36,23.27,13.88,14.23,holt_winters,9.38,14.86
PRODUCE,39.56,10.37,22.46,9.22,8.04,8.09,holt_winters,1.19,9.53


In [15]:
fig = go.Figure()
for m in ["seasonal_naive", "holt_winters", "lightgbm A"]:
    fig.add_bar(name=m, x=fam.index, y=fam[m].round(2), marker_color=MODEL_COLOR[m])
fig.update_layout(**LAYOUT, barmode="group", bargap=0.25, title="Holdout WAPE % by family: LightGBM A vs the two best simple models",
                  yaxis_title="WAPE %", xaxis_title=None)
fig.show()

**Reading it:** LightGBM A has the lowest holdout WAPE in every family. The size of the win varies a lot:

- **Big wins** in GROCERY I, CLEANING and PERSONAL CARE, where Holt-Winters and seasonal naive miss by far more.
- **Small wins** in PRODUCE, DAIRY and BREAD/BAKERY, where Holt-Winters is within about 1-2 pp: these families have a
  stable weekly shape that a per-series smoothing model already captures well.
- **CLEANING** is the hardest family in the holdout (22 % vs about 13 % on selection), and every model struggled
  there; its best simple model was the moving average. FROZEN FOODS stays hard, as expected from notebook 01.

## 9. By horizon (1 to 14 days ahead)

In [16]:
hz = table(hold, ["horizon"])
fig = go.Figure()
for m in MODEL_ORDER:
    fig.add_scatter(x=hz.index, y=hz[m].round(2), name=m, mode="lines+markers", line=dict(color=MODEL_COLOR[m], width=2),
                    marker=dict(size=7), line_dash="dot" if m == "lightgbm B" else "solid")
fig.update_layout(**{**LAYOUT, "height": 420}, title="Holdout WAPE % by forecast horizon (days ahead)",
                  xaxis=dict(title="horizon (days)", dtick=1), yaxis_title="WAPE %")
fig.show()
print("LightGBM A has the lowest WAPE at", int((hz.drop(columns="lightgbm B").idxmin(axis=1) == "lightgbm A").sum()), "of 14 horizons")
print(f"LightGBM A: day 1 {hz.loc[1,'lightgbm A']:.1f}% -> mean of days 8-14 {hz.loc[8:14,'lightgbm A'].mean():.1f}%")
hz.round(1).T

LightGBM A has the lowest WAPE at 14 of 14 horizons
LightGBM A: day 1 10.1% -> mean of days 8-14 13.1%


horizon,1,2,3,4,5,6,7,8,9,10,11,12,13,14
mv,,,,,,,,,,,,,,
naive,28.3,29.2,28.7,30.5,37.0,27.4,23.8,35.1,33.4,29.0,30.6,35.9,27.1,22.3
seasonal_naive,17.0,15.9,15.3,19.2,20.2,16.3,23.8,16.1,18.0,17.3,20.2,21.3,16.9,22.3
moving_average,21.8,34.7,18.9,17.0,26.5,12.8,25.0,21.4,35.6,19.4,15.3,23.8,14.1,26.7
holt_winters,11.3,15.6,16.0,17.2,17.8,15.2,19.5,18.0,20.1,18.2,19.4,18.6,15.3,18.4
lightgbm A,10.1,12.7,12.0,12.0,13.4,11.0,12.7,11.1,13.6,13.5,13.3,14.7,11.7,13.5
lightgbm B,10.7,13.2,12.0,12.1,13.5,11.2,12.7,11.4,14.1,14.0,13.4,14.5,11.7,13.7


**Reading it:** error grows only gently with the horizon for the good models, because the weekly pattern - the
biggest source of predictability - is equally known 1 or 14 days ahead. Naive and seasonal naive have a saw-tooth:
on days 7 and 14 the target falls on the origin's weekday, so "repeat today" and "repeat last week" become the same
forecast. The day-to-day wiggles are mostly calendar (which weekdays and holidays fall at each horizon across only
six origins), not a model property.

## 10. Week-1 vs week-2 buckets: how a planner is measured

Replenishment is planned in weekly quantities, so daily errors that cancel within a week don't hurt. Here each
series-origin's 14 days are summed into a **week-1 total (days 1-7)** and a **week-2 total (days 8-14)**
(`src.metrics.bucket_totals`), and WAPE is computed on those totals.

In [17]:
bk = M.bucket_totals(hold, ["mv", "origin", "series_id"])
bkt = M.grouped_metrics(bk, ["bucket", "mv"]).pivot(index="bucket", columns="mv", values="wape")[MODEL_ORDER]
daily = table(hold.assign(bucket=np.where(hold.horizon <= 7, "week1", "week2")), ["bucket"])
out = pd.concat({"weekly-total WAPE": bkt, "daily WAPE": daily}, axis=0)
out.round(2)

mv                        naive  seasonal_naive  moving_average  holt_winters  lightgbm A  lightgbm B
                  bucket                                                                             
weekly-total WAPE week1   23.84           10.61            8.53         11.43        7.42        7.66
                  week2   25.95           12.98           10.67         14.15        9.39        9.68
daily WAPE        week1   29.71           18.44           22.14         16.22       12.04       12.24
                  week2   30.66           19.03           21.92         18.26       13.11       13.27

**Reading it:** at the weekly-total level the errors are much smaller than daily (day-to-day misses partly cancel),
and week 2 is harder than week 1 for every model. LightGBM A stays the best in both buckets.
The ranking of the simple models changes: the **moving average becomes the second-best model** for weekly totals
(its flat line has no weekday shape, which a weekly total does not need), and seasonal naive also overtakes
Holt-Winters. A planner measured on weekly buckets would find simple models much closer to the ML model than the
daily view suggests.

## 11. Promo days vs non-promo days (association, not causation)

A row counts as a **promo day** if at least one item in that store-family was on promotion (`onpromotion > 0`).

In [18]:
hp = hold.assign(promo=np.where(hold.onpromotion > 0, "promo day", "no promo"))
n = hp[hp.mv == "lightgbm A"].groupby("promo").size().rename("rows (per model)")
pt = table(hp, ["promo"]).join(n)
display(pt.round(2))
share = hp[hp.mv == "lightgbm A"].groupby("family").apply(lambda g: (g.onpromotion > 0).mean() * 100).rename("% promo days")
share.loc[C.FAMILIES].round(1).to_frame().T

,naive,seasonal_naive,moving_average,holt_winters,lightgbm A,lightgbm B,rows (per model)
promo,,,,,,,
no promo,24.93,18.23,22.50,16.85,12.50,10.85,652
promo day,30.27,18.74,22.02,17.24,12.57,12.78,7412


family,BEVERAGES,DAIRY,BREAD/BAKERY,GROCERY I,FROZEN FOODS,CLEANING,PERSONAL CARE,PRODUCE
% promo days,99.6,99.6,79.0,99.6,59.0,99.6,99.6,99.3


**Reading it:** promotions are almost permanent in this holdout: about nine rows in ten are promo days, and six
families have at least one promoted item on more than 99 % of days, so the "no promo" group is small (652 rows per
model) and mostly BREAD/BAKERY and FROZEN FOODS. For LightGBM A, Holt-Winters and seasonal naive the WAPE differs by
less than 0.5 pp between the groups. Naive does worse on promo days, and LightGBM B does *better* on no-promo days -
but with so few no-promo rows, from two families, these are weak signals. The more useful contrast is between promo
variants A and B (section 13). Neither comparison measures promotional uplift: there is no price or discount data.

## 12. Holiday days vs normal days

`is_holiday` marks a day off for that store (national, regional or local, matched to the store's location).
The holdout contains only a handful of such days, so this is a small-sample check.

In [19]:
hd = panel[(panel.is_holiday == 1) & panel.date.between(cal["target_windows"]["holdout"][0], cal["target_windows"]["holdout"][1])]
hd_days = hd.groupby("date").agg(stores=("store_nbr", lambda s: sorted(s.unique())), national=("is_national_holiday", "max"))
display(hd_days)
hh = hold.assign(day=np.where(hold.is_holiday == 1, "holiday", "normal"))
n = hh[hh.mv == "lightgbm A"].groupby("day").size().rename("rows (per model)")
table(hh, ["day"]).join(n).round(2)

,stores,national
date,,
2017-07-03,[54],0
2017-07-24,"[29, 51]",0
2017-07-25,"[29, 51]",0
2017-08-11,"[12, 14, 27, 29, 31, 36, 38, 39, 49, 51, 53, 54]",1
2017-08-15,[14],0


,naive,seasonal_naive,moving_average,holt_winters,lightgbm A,lightgbm B,rows (per model)
day,,,,,,,
holiday,30.46,19.31,16.41,19.97,17.13,18.98,176
normal,30.17,18.71,22.16,17.17,12.47,12.61,7888


**Reading it:** holiday rows are a small share of the holdout (176 rows per model: one national day, 11 August, plus
local/regional days in a few stores). LightGBM A is clearly worse on those days than on normal days, and **it is not
the best model on them: the moving average has the lowest holiday WAPE**. With only 22 holiday store-days this is
indicative, but it fits notebook 01: holiday effects vary a lot from one holiday to the next, so a model that has seen
few comparable days cannot learn them reliably, and a smooth average is hurt less by a day that breaks the weekly pattern.

## 13. Promo variant A (known plan) vs B (no future promo info)

Same model, same settings; the only difference is whether the planned promoted-item count for the target date is a
feature. This is the business question "is it worth getting the promo calendar into the forecast?".

In [20]:
ab = bt[(bt.model == "lightgbm")]
ab = evaluable(ab)
stage_ab = M.grouped_metrics(ab, ["stage", "variant"]).pivot(index="stage", columns="variant", values=["wape", "bias"])
stage_ab = stage_ab.loc[["selection", "calibration", "holdout"]]
stage_ab[("wape", "B - A (pp)")] = stage_ab[("wape", "B")] - stage_ab[("wape", "A")]
display(stage_ab.round(3))
fam_ab = table(hold, ["family"], ["lightgbm A", "lightgbm B"]).loc[C.FAMILIES]
fam_ab["B - A (pp)"] = fam_ab["lightgbm B"] - fam_ab["lightgbm A"]
fam_ab["% promo days"] = share.loc[C.FAMILIES].values
fam_ab.round(2)

wape           bias              wape
variant           A       B      A      B B - A (pp)
stage                                               
selection    11.884  12.047 -0.535 -1.179      0.164
calibration  12.651  12.914 -2.385 -2.224      0.263
holdout      12.571  12.748 -0.458 -0.803      0.177

mv,lightgbm A,lightgbm B,B - A (pp),% promo days
family,,,,
BEVERAGES,17.74,17.92,0.18,99.60
DAIRY,9.15,9.03,-0.12,99.60
BREAD/BAKERY,10.92,10.69,-0.23,78.97
GROCERY I,8.63,9.02,0.39,99.60
FROZEN FOODS,18.66,18.10,-0.56,59.03
CLEANING,22.03,22.11,0.08,99.60
PERSONAL CARE,13.88,14.23,0.35,99.60
PRODUCE,8.04,8.09,0.05,99.31


**Reading it:** knowing the promo plan lowers WAPE in **all three stages**, by a small but consistent margin (a few
tenths of a point overall). By family the picture is mixed: in the holdout the plan helps most in some families and
makes almost no difference, or slightly hurts, in others (see the `B - A` column): the plan helps most in GROCERY I
and PERSONAL CARE, while variant B is slightly better in FROZEN FOODS, BREAD/BAKERY and DAIRY. That is consistent with
notebook 01, where the promoted-item count had only a small association with sales for most families once store size
and month were controlled. A planner would read this as: feed the promo calendar in, it helps on average, but it
is not the main source of LightGBM's advantage.

## 14. What did LightGBM pay attention to?

Mean **gain share** per origin (% of the total split gain), averaged over the 18 selection + calibration origins
(`outputs/lgbm_feature_importance.json`). Gain is "how much the splits on this feature reduced the training loss";
it describes the model, not cause and effect.

In [21]:
imp = pd.concat([pd.DataFrame(fi["variants"][v]).assign(variant=v) for v in ["A", "B"]])
top = imp.groupby("feature").gain_share_pct.max().sort_values(ascending=False).head(15).index
p = imp[imp.feature.isin(top)]
fig = px.bar(p, x="gain_share_pct", y="feature", color="variant", barmode="group", orientation="h",
             color_discrete_map={"A": MODEL_COLOR["lightgbm A"], "B": MODEL_COLOR["lightgbm B"]},
             category_orders={"feature": list(top)}, title="LightGBM feature importance (mean gain share %, top 15)")
fig.update_layout(**{**LAYOUT, "height": 520}, xaxis_title="gain share %", yaxis_title=None)
fig.show()
imp.pivot(index="feature", columns="variant", values="gain_share_pct").loc[top].round(2).T

feature,dow,store_nbr,dom,family_code,promo_t,month,sw_mean4,promo_t_rel28,roll_mean_28,roll_mean_56,sw_last,roll_std_28,cluster,promo_roll_28,roll_std_56
variant,,,,,,,,,,,,,,,
A,19.10,13.87,12.29,7.60,7.96,6.89,3.72,3.62,2.77,2.23,2.48,2.29,1.05,1.74,1.38
B,25.09,14.24,13.47,8.11,NaN,7.19,4.46,NaN,2.39,2.70,2.17,2.03,1.88,1.72,1.66


**Reading it:** because the level is already handled by the scale-free target, the trees spend their effort on
**shape**: day of week (`dow`), the store (`store_nbr`) and family profile, day of month (`dom`: start-of-month and
payday rhythm, see notebook 01), and month. In variant A, the planned promo count (`promo_t` and its ratio to the
recent promo level, `promo_t_rel28`) together take a large share; in B that gain is redistributed mostly to weekday
and history features. Plain lags carry little gain: the same-weekday features (`sw_mean4`, `sw_last`) and rolling
means do that job better.

## 15. Where ML won, why - and where simple models were competitive

The table below asks the question per series: on the holdout, which of the five variant-A models had the lowest WAPE
for each of the 96 series?

In [22]:
ser = table(hold, ["family", "series_id"], C.MODELS[:-1] + ["lightgbm A"])
ser["winner"] = ser.idxmin(axis=1)
wins = ser.groupby(level="family").winner.value_counts().unstack(fill_value=0).reindex(C.FAMILIES).fillna(0).astype(int)
wins.loc["ALL"] = wins.sum()
display(wins)
close = ser[ser.winner != "lightgbm A"].copy()
close["gap to LightGBM A (pp)"] = close["lightgbm A"] - close[C.MODELS[:-1]].min(axis=1)
print(f"Series where a simple model beat LightGBM A on the holdout: {len(close)} of {len(ser)}")
close.sort_values("gap to LightGBM A (pp)", ascending=False).round(2).head(12)

winner,holt_winters,lightgbm A,moving_average,seasonal_naive
family,,,,
BEVERAGES,4,8,0,0
DAIRY,3,9,0,0
BREAD/BAKERY,1,10,1,0
GROCERY I,0,12,0,0
FROZEN FOODS,0,11,0,1
CLEANING,0,11,1,0
PERSONAL CARE,0,12,0,0
PRODUCE,2,10,0,0
ALL,10,83,2,1


Series where a simple model beat LightGBM A on the holdout: 13 of 96


,mv,naive,seasonal_naive,moving_average,holt_winters,lightgbm A,winner,gap to LightGBM A (pp)
family,series_id,,,,,,,
BEVERAGES,54|BEVERAGES,38.16,47.95,41.27,35.23,40.95,holt_winters,5.72
FROZEN FOODS,38|FROZEN FOODS,26.68,21.26,21.75,37.46,24.24,seasonal_naive,2.98
DAIRY,14|DAIRY,16.30,13.78,18.45,12.01,13.70,holt_winters,1.69
BREAD/BAKERY,39|BREAD/BAKERY,12.15,13.42,16.11,11.60,13.26,holt_winters,1.66
DAIRY,54|DAIRY,24.23,15.51,20.62,11.00,12.55,holt_winters,1.55
PRODUCE,12|PRODUCE,20.67,11.94,11.79,9.03,10.37,holt_winters,1.34
BEVERAGES,36|BEVERAGES,19.06,17.75,20.85,16.34,17.50,holt_winters,1.15
DAIRY,12|DAIRY,20.46,13.62,18.51,13.04,14.03,holt_winters,0.99
BEVERAGES,39|BEVERAGES,30.24,24.90,23.93,23.11,24.09,holt_winters,0.98


In [23]:
# Horizon 1 and weekly totals: the two places where simple models came closest
gap_h = (hz[["holt_winters", "seasonal_naive"]].min(axis=1) - hz["lightgbm A"]).rename("best simple - LightGBM A (pp)")
print("Smallest daily gaps by horizon:")
display(gap_h.sort_values().head(4).round(2).to_frame().T)
cal_b = M.grouped_metrics(evaluable(bt[(bt.stage == "calibration") & (bt.mv == "lightgbm A")]), "family").set_index("family").bias
hold_b = M.grouped_metrics(hold[hold.mv == "lightgbm A"], "family").set_index("family").bias
print("LightGBM A bias % by family, calibration vs holdout:")
pd.DataFrame({"calibration": cal_b, "holdout": hold_b}).loc[C.FAMILIES].round(2).T

Smallest daily gaps by horizon:


horizon,1,2,3,13
best simple - LightGBM A (pp),1.21,2.89,3.3,3.61


LightGBM A bias % by family, calibration vs holdout:


family,BEVERAGES,DAIRY,BREAD/BAKERY,GROCERY I,FROZEN FOODS,CLEANING,PERSONAL CARE,PRODUCE
calibration,-3.41,1.00,-2.78,-3.13,-0.21,-1.82,-1.49,-1.38
holdout,-2.42,4.02,0.35,0.31,3.92,-6.31,-0.75,2.28


**Reading it - why ML won:**

- **It pools 96 series.** Holt-Winters fits each series alone on one year. LightGBM learns the weekday, day-of-month
  and holiday shapes once from all stores and families, so a quiet series borrows strength from busy ones.
- **It knows the calendar ahead of time.** Payday / start-of-month (`dom`), holidays and the promo plan are known for
  the target date; the smoothing models only see past sales.
- **It adapts its level at every origin.** The scale-free target re-anchors to the last 28 days, which keeps bias low
  without a trend term.
- The gains are largest in **GROCERY I, PERSONAL CARE and CLEANING**, where week-to-week shape and promo plans move
  more than in the steady fresh families.

**Where simple models were competitive:**

- **Per series:** a simple model beat LightGBM A on 13 of 96 series, 10 of them Holt-Winters, mostly in BEVERAGES,
  DAIRY and PRODUCE. In PRODUCE, DAIRY and BREAD/BAKERY Holt-Winters is within about 1-2 pp at family level: stable
  weekly shapes don't need a global model.
- **FROZEN FOODS:** seasonal naive, not Holt-Winters, is the best simple model, in selection and holdout alike.
- **Holiday days:** the moving average beat LightGBM A on the 176 holiday rows (section 12).
- **Weekly totals:** the moving average and seasonal naive come much closer at the weekly level (section 10),
  though LightGBM A is still better.
- **Day 1:** the gap between Holt-Winters and LightGBM is narrowest at horizon 1 (about 1.2 pp).

**Honesty notes:** the overall LightGBM under-forecast seen on calibration (about −2.4 %) did not carry into the
holdout (−0.46 % overall), but family biases on the holdout range from about −6 % (CLEANING) to +4 % (DAIRY,
FROZEN FOODS): the near-zero total hides offsetting family errors. Nothing was re-tuned for this, because tuning on
calibration would contaminate the intervals and Trust Index fitted there. The holdout is 6 origins in one summer (late June - mid August 2017), with no
December: FROZEN FOODS' hardest season is not tested.

## 16. Summary

In [24]:
lgA, hw_, sn = lb.loc["lightgbm A"], lb.loc["holt_winters"], lb.loc["seasonal_naive"]
summary = pd.DataFrame([
    ["Champion (all 8 families, chosen on selection)", "LightGBM, variant A"],
    ["Holdout WAPE - LightGBM A / Holt-Winters / seasonal naive", f"{lgA.wape:.2f}% / {hw_.wape:.2f}% / {sn.wape:.2f}%"],
    ["Holdout bias - LightGBM A", f"{lgA.bias:+.2f}%"],
    ["Families where LightGBM A had the lowest holdout WAPE", f"{int((fam[C.MODELS[:-1]].min(axis=1) > fam['lightgbm A']).sum())} of 8"],
    ["Series where a simple model beat LightGBM A", f"{len(close)} of {len(ser)}"],
    ["Known promo plan (A) vs none (B), holdout WAPE", f"{lb.loc['lightgbm A','wape']:.2f}% vs {lb.loc['lightgbm B','wape']:.2f}%"],
    ["Weekly-total WAPE, LightGBM A (week 1 / week 2)", f"{bkt.loc['week1','lightgbm A']:.2f}% / {bkt.loc['week2','lightgbm A']:.2f}%"],
], columns=["what", "result"]).set_index("what")
summary

,result
what,
"Champion (all 8 families, chosen on selection)","LightGBM, variant A"
Holdout WAPE - LightGBM A / Holt-Winters / seasonal naive,12.57% / 17.23% / 18.73%
Holdout bias - LightGBM A,-0.46%
Families where LightGBM A had the lowest holdout WAPE,8 of 8
Series where a simple model beat LightGBM A,13 of 96
"Known promo plan (A) vs none (B), holdout WAPE",12.57% vs 12.75%
"Weekly-total WAPE, LightGBM A (week 1 / week 2)",7.42% / 9.39%


Notebook 03 takes the champion forecasts and asks the next planner question: **how uncertain is each forecast, and
which ones deserve a human look?**